# スコアベースモデルと拡散モデル

拡散モデルは、データを少しずつノイズへ壊す過程を先に決め、逆向きにノイズを取り除く関数を推定します。スコアベースモデルは、各時刻の分布で log p(x) の勾配を推定します。この勾配は、いまの点をどちらへ動かすとデータらしい領域へ近づくかを示す方向です。

## 1次元の二峰分布で全体像を見る

画像では見えにくい現象も、1次元なら数値で追えます。左と右に山を持つ分布を使い、ノイズを足すと山がぼやけ、逆向きの更新で山へ戻る様子を確認します。

In [ ]:
import math
import random
from statistics import mean, pstdev

random.seed(41)

MIXTURE = [
    {'weight': 0.58, 'mu': -2.1, 'sigma': 0.45},
    {'weight': 0.42, 'mu': 1.9, 'sigma': 0.55},
]


def sample_data(n, rng=random):
    xs = []
    for _ in range(n):
        comp = MIXTURE[0] if rng.random() < MIXTURE[0]['weight'] else MIXTURE[1]
        xs.append(rng.gauss(comp['mu'], comp['sigma']))
    return xs


def describe(xs):
    return {
        'mean': mean(xs),
        'std': pstdev(xs),
        'left': sum(x < 0 for x in xs) / len(xs),
        'right': sum(x >= 0 for x in xs) / len(xs),
    }


def show_stats(name, xs):
    st = describe(xs)
    print(name, 'mean=', round(st['mean'], 3), 'std=', round(st['std'], 3), 'left/right=', round(st['left'], 3), round(st['right'], 3))

data = sample_data(3000)
show_stats('data', data)

## スコアは密度の上り方向

スコアは d log p(x) / dx です。山の左側では右へ、山の右側では左へ向きます。二峰分布では中央付近の向きが混合比と分散に依存するため、単純な平均方向だけでは説明できません。

In [ ]:
def normal_pdf(x, mu, sigma):
    z = (x - mu) / sigma
    return math.exp(-0.5 * z * z) / (math.sqrt(2 * math.pi) * sigma)


def mixture_pdf(x, noise_sigma=0.0):
    total = 0.0
    for comp in MIXTURE:
        sigma = math.sqrt(comp['sigma'] ** 2 + noise_sigma ** 2)
        total += comp['weight'] * normal_pdf(x, comp['mu'], sigma)
    return total


def mixture_score(x, noise_sigma=0.0):
    numer = 0.0
    denom = 0.0
    for comp in MIXTURE:
        sigma2 = comp['sigma'] ** 2 + noise_sigma ** 2
        sigma = math.sqrt(sigma2)
        p = comp['weight'] * normal_pdf(x, comp['mu'], sigma)
        numer += p * (-(x - comp['mu']) / sigma2)
        denom += p
    return numer / max(denom, 1e-300)

for x in [-3, -2, -1, 0, 1, 2, 3]:
    s = mixture_score(x)
    print('x=', f'{x:>2}', 'score=', round(s, 3), 'move=', 'right' if s > 0 else 'left')

## ノイズを足すと分布はなめらかになる

拡散の前向き過程は、データへ段階的にノイズを足します。1次元では、元分布にガウスノイズを畳み込んだ分布になります。ノイズが大きいほど山はつぶれ、スコアの向きも緩やかになります。

In [ ]:
def add_noise(xs, sigma, seed=0):
    rng = random.Random(seed)
    return [x + sigma * rng.gauss(0.0, 1.0) for x in xs]

for sigma in [0.2, 0.8, 1.8, 3.0]:
    noisy = add_noise(data, sigma, seed=int(sigma * 100))
    show_stats(f'noise sigma={sigma}', noisy)

print('scores at x=0')
for sigma in [0.0, 0.5, 1.5, 3.0]:
    print('sigma=', sigma, 'score=', round(mixture_score(0.0, noise_sigma=sigma), 4))

## DSM はノイズ付き点の戻る向きを学ぶ

Denoising Score Matching では、x_t = x_0 + sigma * eps を作り、target = -(x_t - x_0) / sigma^2 を教師にします。個々の教師は揺れますが、同じ x_t 付近で平均すると、ノイズでぼかされた分布のスコアへ近づきます。

In [ ]:
def dsm_targets_near(query_x, sigma, n=5000, width=0.08, seed=11):
    rng = random.Random(seed)
    vals = []
    tries = 0
    while len(vals) < n and tries < n * 200:
        tries += 1
        x0 = sample_data(1, rng)[0]
        eps = rng.gauss(0.0, 1.0)
        xt = x0 + sigma * eps
        if abs(xt - query_x) <= width:
            vals.append(-(xt - x0) / (sigma * sigma))
    return vals

for query_x in [-2.0, 0.0, 2.0]:
    vals = dsm_targets_near(query_x, sigma=0.8, n=800, width=0.12, seed=20 + int(query_x * 10))
    empirical = mean(vals)
    analytic = mixture_score(query_x, noise_sigma=0.8)
    print('x=', query_x, 'DSM avg=', round(empirical, 3), 'analytic score=', round(analytic, 3), 'samples=', len(vals))

## Langevin 更新でスコアからサンプルを作る

スコアが分かると、x をスコア方向へ少し動かし、同時に小さなノイズを加えることで分布からサンプルできます。大きなノイズ幅から小さなノイズ幅へ下げる annealed Langevin dynamics は、粗い形から細部へ戻る発想です。

In [ ]:
def langevin_sample(n=2400, sigmas=(3.0, 1.5, 0.8, 0.35, 0.12), steps_per_sigma=35, step_scale=0.055, seed=77):
    rng = random.Random(seed)
    xs = [rng.gauss(0.0, sigmas[0]) for _ in range(n)]
    for sigma in sigmas:
        step = step_scale * sigma * sigma
        noise_scale = math.sqrt(2 * step)
        for _ in range(steps_per_sigma):
            xs = [
                max(-6.0, min(6.0, x + step * mixture_score(x, noise_sigma=sigma) + noise_scale * rng.gauss(0.0, 1.0)))
                for x in xs
            ]
    return xs

langevin = langevin_sample()
show_stats('langevin', langevin)
show_stats('target ', data)

## DDPM はノイズ予測として同じ方向を表す

DDPM では x_t = sqrt(alpha_bar_t) x_0 + sqrt(1 - alpha_bar_t) eps と書き、モデルは eps を予測します。スコアとノイズ予測は同じ戻り方向の別表現です。eps が分かれば x_0 の推定値を作れます。

In [ ]:
def make_schedule(T=40, beta_start=0.001, beta_end=0.11):
    betas = []
    for t in range(T):
        r = t / (T - 1) if T > 1 else 0.0
        betas.append(beta_start + (beta_end - beta_start) * r)
    alphas = [1.0 - b for b in betas]
    alpha_bars = []
    prod = 1.0
    for a in alphas:
        prod *= a
        alpha_bars.append(prod)
    return betas, alphas, alpha_bars

T = 40
betas, alphas, alpha_bars = make_schedule(T)
print('alpha_bar first/last:', round(alpha_bars[0], 4), round(alpha_bars[-1], 4))


def q_sample(x0, t, rng):
    eps = rng.gauss(0.0, 1.0)
    ab = alpha_bars[t]
    xt = math.sqrt(ab) * x0 + math.sqrt(1.0 - ab) * eps
    return xt, eps

rng = random.Random(5)
for t in [0, 10, 25, 39]:
    xt, eps = q_sample(-2.1, t, rng)
    print('t=', t, 'xt=', round(xt, 3), 'eps=', round(eps, 3))

## ノイズ予測器を閉形式で近似する

小さな例なので、線形予測器 eps_hat = a_t x_t + b_t を時刻ごとに最小二乗で当てます。実際の画像生成では U-Net などがこの役割を担います。各時刻で「混ざったノイズ量」を推定し、その推定を使って逆向きに少しずつ戻します。

In [ ]:
def fit_linear_eps_predictor(dataset, T, samples_per_t=900, seed=101):
    rng = random.Random(seed)
    params = []
    for t in range(T):
        xs = []
        ys = []
        for _ in range(samples_per_t):
            x0 = dataset[rng.randrange(len(dataset))]
            xt, eps = q_sample(x0, t, rng)
            xs.append(xt)
            ys.append(eps)
        mx = mean(xs)
        my = mean(ys)
        varx = mean((x - mx) ** 2 for x in xs)
        cov = mean((x - mx) * (y - my) for x, y in zip(xs, ys))
        a = 0.0 if varx == 0 else cov / varx
        b = my - a * mx
        mse = mean((a * x + b - y) ** 2 for x, y in zip(xs, ys))
        params.append((a, b, mse))
    return params

params = fit_linear_eps_predictor(data, T)
for t in [0, 5, 15, 25, 39]:
    a, b, mse = params[t]
    print('t=', t, 'a=', round(a, 3), 'b=', round(b, 3), 'mse=', round(mse, 4))

## 逆過程でノイズから戻す

学習済みの eps_hat を使い、DDPM の平均更新で x_T から x_0 へ戻します。予測器が低容量なので完全には戻りませんが、左右の山へ質量が分かれることを確認できます。

In [ ]:
def eps_hat(x, t, params):
    a, b, _ = params[t]
    return a * x + b


def reverse_ddpm(params, n=2400, skip=1, seed=202):
    rng = random.Random(seed)
    xs = [rng.gauss(0.0, 1.0) for _ in range(n)]
    t_values = list(range(T - 1, -1, -skip))
    if t_values[-1] != 0:
        t_values.append(0)
    for t in t_values:
        beta = betas[t]
        alpha = alphas[t]
        ab = alpha_bars[t]
        next_xs = []
        for x in xs:
            pred_eps = eps_hat(x, t, params)
            mean_x = (x - beta * pred_eps / math.sqrt(max(1e-8, 1.0 - ab))) / math.sqrt(alpha)
            if t > 0:
                mean_x += math.sqrt(beta) * rng.gauss(0.0, 1.0)
            next_xs.append(max(-6.0, min(6.0, mean_x)))
        xs = next_xs
    return xs

samples = reverse_ddpm(params, skip=1)
show_stats('ddpm skip=1', samples)
show_stats('target     ', data)

## ステップを粗くすると品質が落ちる

逆過程は小さな修正の積み重ねです。ステップを間引くと、局所的なノイズ予測が同じでも最終分布は崩れます。高速化では、ステップ数、分散、モード比率、下流評価を同時に見ます。

In [ ]:
for skip in [1, 2, 4, 8]:
    xs = reverse_ddpm(params, skip=skip, seed=300 + skip)
    st = describe(xs)
    print('skip=', skip, 'mean=', round(st['mean'], 3), 'std=', round(st['std'], 3), 'left/right=', round(st['left'], 3), round(st['right'], 3))
print('target left/right:', round(describe(data)['left'], 3), round(describe(data)['right'], 3))

## 局所誤差と生成品質は別に見る

eps の MSE が小さくても、逆過程の積み重ねで分布がずれることがあります。評価では、時刻ごとのノイズ予測誤差、生成サンプルの統計、モードの被覆、速度を分けて確認します。

In [ ]:
def evaluate_eps_mse(params, dataset, t, n=600, seed=500):
    rng = random.Random(seed + t)
    losses = []
    for _ in range(n):
        x0 = dataset[rng.randrange(len(dataset))]
        xt, eps = q_sample(x0, t, rng)
        losses.append((eps_hat(xt, t, params) - eps) ** 2)
    return mean(losses)

for t in [0, 5, 10, 20, 30, 39]:
    print('t=', t, 'mse=', round(evaluate_eps_mse(params, data, t), 4), 'alpha_bar=', round(alpha_bars[t], 4))

スコアベースモデルは「戻る向き」を直接推定し、DDPM は「取り除くノイズ」を推定します。どちらも、前向きに壊す過程を固定し、逆向きにデータ分布へ戻るための方向情報を使う設計です。良い生成器にするには、局所的な予測誤差だけでなく、逆過程を通した最終分布の形まで検証します。